# Model Optimization: Baseline Evaluation

In this notebook, we'll establish baseline performance metrics for our models using SageMaker endpoints. We'll deploy each model to a SageMaker endpoint and measure its performance characteristics, which will serve as a baseline for our optimization techniques.

## What are Baseline Metrics?

Baseline metrics provide a reference point for measuring the effectiveness of our optimization techniques. By establishing these metrics first, we can quantify the improvements achieved through quantization, pruning, and knowledge distillation.

### Key Metrics We'll Measure:
- **Model Size**: How much storage space the model requires
- **Inference Time**: How long it takes to generate predictions
- **Number of Parameters**: How many trainable parameters the model has

### SageMaker Endpoint Approach
This notebook uses SageMaker endpoints to host our models and perform inference. This approach allows us to:
1. Deploy models to dedicated instances with appropriate resources
2. Measure inference performance in a production-like environment
3. Scale to larger models that might not fit on the notebook instance
4. Learn the workflow for deploying and optimizing models in production

## 1. Import Dependencies

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import sagemaker
import re
from sagemaker.huggingface import HuggingFaceModel
from sagemaker import get_execution_role
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
from transformers import AutoModelForMaskedLM
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

# Define the instance type to use for model endpoints
ENDPOINT_INSTANCE_TYPE = "ml.m5.xlarge"  # Good balance of CPU and memory for inference
%store ENDPOINT_INSTANCE_TYPE

## 3. Define Models to Evaluate

We'll define a set of models to evaluate, covering different tasks and model sizes.

In [ ]:
# Define models to evaluate
model_info = {
    "sentiment-analysis": {
        "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
        "task": "text-classification",
        "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english"
    },
    "ner": {
        "model_name": "dbmdz/bert-large-cased-finetuned-conll03-english",
        "task": "token-classification",
        "hub_model_id": "dbmdz/bert-large-cased-finetuned-conll03-english"
    },
    "question-answering": {
        "model_name": "distilbert-base-cased-distilled-squad",
        "task": "question-answering",
        "hub_model_id": "distilbert-base-cased-distilled-squad"
    },
    "masked-lm": {
        "model_name": "distilroberta-base",
        "task": "fill-mask",
        "hub_model_id": "distilroberta-base"
    }
}

# Save model information to a file
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Defined {len(model_info)} models for evaluation")

## 4. Define Sample Inputs for Each Task

We'll define sample inputs for each task to use for inference testing.

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment-analysis": {
        "inputs": "I really enjoyed this movie. The acting was superb and the plot was engaging."
    },
    "ner": {
        "inputs": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington."
    },
    "question-answering": {
        "inputs": {
            "question": "What is machine learning?",
            "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
        }
    },
    "masked-lm": {
        "inputs": "The <mask> is a large language model trained by OpenAI."
    }
}

## 5. Helper Functions for Model Evaluation

We'll define helper functions to measure model size, parameter count, and inference time.

In [ ]:
def sanitize_name(name):
    """Sanitize a name to be used as part of an endpoint name.
    Endpoint names must satisfy regex pattern: [a-zA-Z0-9](-*[a-zA-Z0-9]){0,62}
    """
    # Replace underscores with hyphens
    sanitized = name.replace('_', '-')
    # Replace any other invalid characters with hyphens
    sanitized = re.sub(r'[^a-zA-Z0-9-]', '-', sanitized)
    # Ensure it doesn't start or end with a hyphen
    sanitized = sanitized.strip('-')
    # Ensure no consecutive hyphens
    sanitized = re.sub(r'-+', '-', sanitized)
    return sanitized

def get_model_size(model_name, task):
    """Calculate model size in MB by loading it locally."""
    # Load the model locally to measure its size
    if task == "text-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "fill-mask":
        model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Calculate model size
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

def get_num_parameters(model_name, task):
    """Calculate number of parameters in the model."""
    # Load the model locally to count parameters
    if task == "text-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "fill-mask":
        model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    return sum(p.numel() for p in model.parameters())

def measure_inference_time(endpoint_name, payload, num_runs=10):
    """Measure average inference time over multiple runs using a SageMaker endpoint."""
    # Create a SageMaker runtime client
    runtime_client = boto3.client('sagemaker-runtime')
    
    # Warm-up run
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(payload)
    )
    
    # Measure inference time
    inference_times = []
    for _ in range(num_runs):
        start_time = time.time()
        response = runtime_client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=json.dumps(payload)
        )
        end_time = time.time()
        inference_times.append((end_time - start_time) * 1000)  # Convert to ms
    
    return sum(inference_times) / len(inference_times)

## 6. Deploy Models to SageMaker Endpoints

Now we'll deploy each model to a SageMaker endpoint for inference.

In [ ]:
# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Dictionary to store endpoint names
endpoint_names = {}

# Deploy each model to a SageMaker endpoint
for model_key, info in model_info.items():
    print(f"Deploying {model_key} model to SageMaker endpoint...")
    
    # Create a unique endpoint name with sanitized model key
    endpoint_name = f"model-opt-workshop-{model_key}-{int(time.time())}"
    endpoint_names[model_key] = endpoint_name
    
    # Create environment variables for the Hugging Face model
    env = {
        'HF_MODEL_ID': info["hub_model_id"],
        'HF_TASK': info["task"]
    }
    
    # Create a Hugging Face model
    huggingface_model = HuggingFaceModel(
        model_data=None,  # No model data, will use HF_MODEL_ID instead
        role=SAGEMAKER_ROLE_ARN,
        transformers_version="4.26",
        pytorch_version="1.13",
        py_version="py39",
        env=env
    )
    
    # Deploy the model to an endpoint
    predictor = huggingface_model.deploy(
        initial_instance_count=1,
        instance_type=ENDPOINT_INSTANCE_TYPE,
        endpoint_name=endpoint_name
    )
    
    print(f"Deployed {model_key} model to endpoint: {endpoint_name}")

# Store endpoint names for later use
%store endpoint_names

## 7. Measure Baseline Metrics

Now we'll measure the baseline metrics for each model.

In [ ]:
# Dictionary to store baseline metrics
baseline_metrics = {}

# Measure metrics for each model
for model_key, info in model_info.items():
    print(f"Measuring metrics for {model_key} model...")
    
    # Get model size and parameter count
    model_size = get_model_size(info["model_name"], info["task"])
    num_parameters = get_num_parameters(info["model_name"], info["task"])
    
    # Measure inference time using the endpoint
    endpoint_name = endpoint_names[model_key]
    payload = sample_inputs[model_key]
    inference_time = measure_inference_time(endpoint_name, payload)
    
    # Store metrics
    baseline_metrics[model_key] = {
        "model_name": info["model_name"],
        "task": info["task"],
        "model_size": round(model_size, 2),
        "num_parameters": num_parameters,
        "inference_time": round(inference_time, 2)
    }
    
    print(f"Model size: {model_size:.2f} MB")
    print(f"Number of parameters: {num_parameters:,}")
    print(f"Average inference time: {inference_time:.2f} ms")
    print("---")

# Save baseline metrics to a file
with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

# Store baseline metrics for later use
%store baseline_metrics

## 8. Analyze Baseline Metrics

Now let's analyze the baseline metrics to understand the performance characteristics of our models.

In [ ]:
# Create a DataFrame with the metrics for better display
model_names = [metrics["model_name"] for metrics in baseline_metrics.values()]
model_sizes = [metrics["model_size"] for metrics in baseline_metrics.values()]
inference_times = [metrics["inference_time"] for metrics in baseline_metrics.values()]
num_parameters = [metrics["num_parameters"] for metrics in baseline_metrics.values()]

# Create a DataFrame
metrics_df = pd.DataFrame({
    "Model": model_names,
    "Size (MB)": model_sizes,
    "Inference Time (ms)": inference_times,
    "Parameters": num_parameters
})

# Display the metrics table
metrics_df

## 9. Visualize Baseline Metrics

Let's create some visualizations to better understand the baseline metrics.

In [ ]:
# Create visualizations
plt.figure(figsize=(15, 10))

# Model size plot
plt.subplot(2, 2, 1)
sns.barplot(x=metrics_df["Model"], y=metrics_df["Size (MB)"])
plt.title("Model Size (MB)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Inference time plot
plt.subplot(2, 2, 2)
sns.barplot(x=metrics_df["Model"], y=metrics_df["Inference Time (ms)"])
plt.title("Inference Time (ms)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Number of parameters plot
plt.subplot(2, 2, 3)
sns.barplot(x=metrics_df["Model"], y=metrics_df["Parameters"])
plt.title("Number of Parameters")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Size vs. Inference Time scatter plot
plt.subplot(2, 2, 4)
sns.scatterplot(x=metrics_df["Size (MB)"], y=metrics_df["Inference Time (ms)"])
plt.title("Size vs. Inference Time")
plt.xlabel("Size (MB)")
plt.ylabel("Inference Time (ms)")
for i, model in enumerate(metrics_df["Model"]):
    plt.annotate(model, (metrics_df["Size (MB)"][i], metrics_df["Inference Time (ms)"][i]))
plt.tight_layout()

plt.show()

## 10. Test Inference with the Endpoints

Let's test inference with each endpoint to make sure they're working correctly.

In [ ]:
# Create a SageMaker runtime client
runtime_client = boto3.client('sagemaker-runtime')

# Test inference with each endpoint
for model_key, info in model_info.items():
    print(f"Testing inference with {model_key} model...")
    
    # Get the endpoint name and payload
    endpoint_name = endpoint_names[model_key]
    payload = sample_inputs[model_key]
    
    # Invoke the endpoint
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(payload)
    )
    
    # Parse the response
    result = json.loads(response['Body'].read().decode())
    
    # Display the result
    print(f"Input: {payload}")
    print(f"Output: {result}")
    print("---")

## 11. Next Steps

Now that we've established baseline metrics for our models, we'll explore quantization techniques in the next notebook to reduce model size and improve inference speed.

### What We've Learned:
- How to deploy models to SageMaker endpoints
- How to measure key performance metrics for transformer models
- How model size relates to inference time
- The baseline performance characteristics of our models

### What's Next - Quantization:
Quantization is a technique that reduces the precision of the numbers used to represent a model's parameters. For example, converting 32-bit floating point numbers to 8-bit integers. This significantly reduces model size and can improve inference speed, often with minimal impact on accuracy.

## 12. Clean Up Resources (Optional)

If you want to clean up the resources created in this notebook, you can delete the endpoints. Note that you might want to keep them running if you plan to compare them with optimized models later.

In [ ]:
# Uncomment and run this cell to delete the endpoints
"""
# Create a SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Delete each endpoint
for model_key, endpoint_name in endpoint_names.items():
    print(f"Deleting endpoint: {endpoint_name}")
    sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Endpoint {endpoint_name} deleted")
"""